# Question 1：Vector Add

工作区根目录的 `vecAdd.cu` 在原始示例基础上修改，保留 `add`、`dim3 grid/tpb` 和原来的内存操作顺序。每个 block 仍使用 **32 个线程**。

先在 Colab 打开本 notebook，再手动把工作区根目录的 **`vecAdd.cu`** 上传到 Colab 的 **`/content`**。按下面顺序运行即可。


### 1. 检查 GPU 和 CUDA


In [66]:
!nvidia-smi
!nvcc --version

Fri Nov  6 00:53:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 2. 确认上传的源码


In [ ]:
%cd /content
!ls -l vecAdd.cu


### 3. 编译

沿用教程中的 `sm_75`（Tesla T4）。修改源码后，重新上传并编译。


In [ ]:
!nvcc -arch=sm_75 vecAdd.cu -o vecAdd


### 4. 测试 N = 512

程序只接收一个参数：向量长度 N。每次运行一次 kernel。

输出包含线程/block 配置、三个阶段的时间和最大误差 `max_error`。误差为 0 表示 CPU/GPU 数值完全相等。


In [ ]:
!./vecAdd 512


### 5. 测试 N = 263149

`grid` 向上取整，kernel 中的 `if (i < N)` 防止越界。


In [ ]:
!./vecAdd 263149


### 6. 导入绘图库


In [ ]:
import io
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


### 7. 测量不同长度

每个长度只测量一次。C++ 源码中的 CPU 计时覆盖 H2D、kernel、D2H；每个阶段结束时等待 GPU 完成。kernel 时间包含启动和同步开销。

结果保存在 Colab 的 `timing_results.csv`，不自动下载。


In [ ]:
lengths = [
    512, 1024, 4096, 16384, 65536, 131072, 263149, 524288,
    1048576, 2097152, 4194304, 6291456,
    8388608, 10485760, 12582912, 16777216,
]
results = []
for N in lengths:
    output = subprocess.check_output(['./vecAdd', str(N)], text=True)
    results.append(pd.read_csv(io.StringIO(output)))

timing = pd.concat(results, ignore_index=True)
timing.to_csv('timing_results.csv', index=False)
display(timing)


### 8. 堆叠柱状图

H2D 包含两个输入向量，D2H 包含一个输出向量。


In [ ]:
timing.set_index('N')[['H2D_ms', 'kernel_ms', 'D2H_ms']].plot(
    kind='bar', stacked=True, figsize=(12, 5)
)
plt.xlabel('Vector length N')
plt.ylabel('Time (ms)')
plt.xticks(rotation=60)
plt.tight_layout()
plt.savefig('timing_breakdown.png')
plt.show()


### 9. 查看 CPU/GPU 误差


In [ ]:
display(timing[['N', 'max_error']])


### 10. 查看 Nsight Compute 的分析模块

与教程一样使用 `ncu`。如果提示找不到命令，需要先在 Colab 配置 Nsight Compute。


In [ ]:
!ncu --list-sections


### 11. N = 512 的 Achieved Occupancy

读取输出中 Occupancy 表格的 **Achieved Occupancy**。


In [ ]:
!ncu --section Occupancy ./vecAdd 512


### 12. N = 263149 的 Achieved Occupancy

如果出现 `ERR_NVGPUCTRPERM`，当前 Colab 不允许访问性能计数器，无法在该运行时测得占用率。前面的计时和绘图单独运行，不使用 Nsight 插桩后的时间。


In [ ]:
!ncu --section Occupancy ./vecAdd 263149
